# Экспорт результатов

FESTIM позволяет экспортировать:

- поля концентрации и температуры;
- одномерные профили;
- производные величины: средние значения, интегралы, минимумы, максимумы и потоки через поверхности.

Для одномерной задачи рассмотрим три наиболее полезных объекта:

- `Profile1DExport` — профиль концентрации;
- `AverageVolume` — средняя концентрация в объёме;
- `SurfaceFlux` — поток через выбранную поверхность.

Результаты доступны непосредственно в Python. Для производных величин их также можно записывать в файлы `.csv`.


## Простая одномерная задача

Рассмотрим нестационарную диффузию в области $0 \leq x \leq 1$:

$$
c(0,t)=1, \qquad c(1,t)=0.
$$

Начальная концентрация по умолчанию равна нулю.


In [ ]:
import festim as F
import numpy as np
import matplotlib.pyplot as plt

model = F.HydrogenTransportProblem()

model.mesh = F.Mesh1D(
    vertices=np.linspace(0.0, 1.0, 101),
)

material = F.Material(
    D_0=0.1,
    E_D=0.0,
)

volume = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, 1.0],
    material=material,
)

left = F.SurfaceSubdomain1D(id=1, x=0.0)
right = F.SurfaceSubdomain1D(id=2, x=1.0)

model.subdomains = [volume, left, right]

H = F.Species("H")
model.species = [H]

model.temperature = 400.0

model.boundary_conditions = [
    F.FixedConcentrationBC(
        subdomain=left,
        value=1.0,
        species=H,
    ),
    F.FixedConcentrationBC(
        subdomain=right,
        value=0.0,
        species=H,
    ),
]

model.settings = F.Settings(
    atol=1e-10,
    rtol=1e-8,
    final_time=10.0,
    stepsize=F.Stepsize(initial_value=0.1),
)

## Профиль концентрации

`Profile1DExport` сохраняет координаты и рассчитанные профили концентрации:

- `profile.x` — координаты;
- `profile.t` — моменты времени;
- `profile.data` — профили концентрации.


In [ ]:
profile = F.Profile1DExport(
    field=H,
    subdomain=volume,
)

## Производные величины

Для примера экспортируем только две величины:

- среднюю концентрацию в области;
- поток через правую поверхность.

Если указан `filename`, значения дополнительно записываются в `.csv`. После расчёта они также доступны через атрибуты `.t` и `.data`.


In [ ]:
average_concentration = F.AverageVolume(
    field=H,
    volume=volume,
    filename="average_concentration.csv",
)

right_flux = F.SurfaceFlux(
    field=H,
    surface=right,
    filename="right_flux.csv",
)

model.exports = [
    profile,
    average_concentration,
    right_flux,
]

## Запуск расчёта


In [ ]:
model.initialise()
model.run()

## Визуализация профиля концентрации

Покажем несколько профилей в разные моменты времени.


In [ ]:
indices = np.unique(
    np.linspace(0, len(profile.t) - 1, 4, dtype=int)
)

plt.figure(figsize=(7, 4))

for index in indices:
    plt.plot(
        profile.x,
        profile.data[index],
        label=f"t = {profile.t[index]:.2f}",
    )

plt.xlabel("Координата")
plt.ylabel("Концентрация H")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Средняя концентрация


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    average_concentration.t,
    average_concentration.data,
)

plt.xlabel("Время")
plt.ylabel("Средняя концентрация H")
plt.grid(alpha=0.3)
plt.show()

## Поток через правую поверхность


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(
    right_flux.t,
    right_flux.data,
)

plt.xlabel("Время")
plt.ylabel("Поток через правую поверхность")
plt.grid(alpha=0.3)
plt.show()

## Экспорт поля в ParaView

Для одномерного анализа обычно удобнее `Profile1DExport`. Если требуется сохранить полное поле в формате VTX и открыть его в ParaView, можно использовать:


In [ ]:
vtx_export = F.VTXSpeciesExport(
    filename="H_concentration.bp",
    field=H,
)

# Для записи файла объект нужно добавить в model.exports
# до вызова model.initialise() и model.run().

## Другие доступные экспорты

FESTIM также предоставляет интегральные, минимальные и максимальные величины по объёму и поверхности, пользовательские выражения, экспорт температурного поля и скоростей реакций.
